# Tome 11, Appendix A: Spectral Graph Theory (Python companion)

Exercises first, solutions at the end. We check Laplacian spectra, count spanning trees, arborescences and Eulerian circuits three ways (brute force, determinants, eigenvalues), sample uniform spanning trees, test Cheeger's inequality against exact edge expansion, cluster graphs with the Fiedler vector, and measure how close random regular graphs come to the Alon–Boppana bound.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math, time
from collections import Counter, defaultdict
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import sympy as sp
random.seed(9); rng = np.random.default_rng(9)

## Exercises

**P9.1 (★) Laplacian spectra.** Check the spectra of Example 9.3 ($K_n$, $C_n$, $Q_n$, Petersen) and Proposition 9.2 (the multiplicity of $0$ counts components) on random graphs.

**P9.2 (★★) Counting trees three ways.** For random small graphs, count spanning trees by brute force, by the determinant $\det L_w$ (exactly, in integers), and by $\frac1n\prod_{i\ge2}\mu_i$ (in floating point). Check $\tau(Q_n)$ (Corollary 9.6) for $n\le6$ and the directed theorem (Theorem 9.4) on random directed multigraphs.

**P9.3 (★★) Uniform spanning trees.** Implement the Aldous–Broder algorithm: run a random walk until it has visited every vertex, and keep the edge by which each vertex was first entered. Check on $K_4$ and on a small grid that every spanning tree comes out with the same frequency.

**P9.4 (★★) Eulerian circuits.** Count Eulerian circuits of random Eulerian directed graphs by backtracking and compare with the BEST theorem (Theorem 9.7). Enumerate de Bruijn sequences for small $q$ and $n$ and compare with Corollary 9.8 and Exercise 9.4.

**P9.5 (★★) Cheeger's inequality.** For random $3$-regular graphs on up to $16$ vertices, compute $h(G)$ exactly by trying all sets, and compare with $\frac{\mu_2}2$, $\sqrt{2d\mu_2}$ and the best sweep set of the Fiedler vector (Theorem 9.9).

**P9.6 (★★) Spectral clustering.** In the *stochastic block model*, $n$ vertices are split into two hidden halves, and each pair is joined with probability $p$ inside a half and $q$ across. How well does the sign of the Fiedler vector recover the halves, as $q$ grows towards $p$?

**P9.7 (★★) How good are random expanders?** For random $d$-regular graphs, compute $\lambda=\max(|\lambda_2|,|\lambda_n|)$ and compare with $2\sqrt{d-1}$. Check the expander mixing lemma (Lemma 9.10) on random sets, and the Alon–Boppana bound (Theorem 9.11) using actual distances between edges.

**P9.8 (★) Random walks forget.** Compare $\|p_t-u\|$ with $(\lambda/d)^t$ (Exercise 9.8) for a random $4$-regular graph and for a cycle of the same size.

## Solutions

### P9.1 Laplacian spectra

In [ ]:
def lap(G):
    return nx.laplacian_matrix(G, nodelist=sorted(G.nodes)).toarray().astype(float)
def spec(G):
    return np.round(np.linalg.eigvalsh(lap(G)), 8)
def show(name, G):
    c = Counter(np.round(spec(G), 6)); print(f'{name:10s}', {float(k): v for k, v in sorted(c.items())})
show('K6', nx.complete_graph(6))
show('Q4', nx.hypercube_graph(4))
show('Petersen', nx.petersen_graph())
n = 9; C = spec(nx.cycle_graph(n))
assert np.allclose(sorted(C), sorted(2 - 2 * np.cos(2 * np.pi * np.arange(n) / n)))
print('C9 matches 2-2cos(2 pi j/9)')
for trial in range(200):
    G = nx.gnp_random_graph(random.randint(5, 25), random.choice([0.05, 0.1, 0.2]), seed=trial)
    assert sum(abs(x) < 1e-8 for x in spec(G)) == nx.number_connected_components(G)
print('multiplicity of 0 = number of components on 200 random graphs')

### P9.2 Counting trees three ways

The determinant is computed exactly with the Bareiss algorithm, which divides only by previous pivots and keeps all entries integers.

In [ ]:
def bareiss_det(M):
    M = [list(map(int, row)) for row in M]; n = len(M); sign = 1; prev = 1
    for k in range(n - 1):
        if M[k][k] == 0:
            for i in range(k + 1, n):
                if M[i][k] != 0: M[k], M[i] = M[i], M[k]; sign = -sign; break
            else: return 0
        for i in range(k + 1, n):
            for j in range(k + 1, n):
                M[i][j] = (M[i][j] * M[k][k] - M[i][k] * M[k][j]) // prev
        prev = M[k][k]
    return sign * M[n - 1][n - 1] if n else 1
def tau_det(G, w=0):
    nodes = sorted(G.nodes); L = nx.laplacian_matrix(G, nodelist=nodes).toarray()
    keep = [i for i in range(len(nodes)) if i != w]
    return bareiss_det(L[np.ix_(keep, keep)])
def tau_brute(G):
    n = G.number_of_nodes(); E = list(G.edges)
    return sum(1 for T in itertools.combinations(E, n - 1) if nx.is_tree(nx.Graph(list(T))) and nx.Graph(list(T)).number_of_nodes() == n)
def tau_eig(G):
    mu = np.linalg.eigvalsh(lap(G)); return float(np.prod(mu[1:]) / len(mu))
for trial in range(30):
    G = nx.gnp_random_graph(random.randint(4, 7), 0.6, seed=100 + trial)
    if not nx.is_connected(G): continue
    b, d, e = tau_brute(G), tau_det(G, w=random.randrange(G.number_of_nodes())), tau_eig(G)
    assert b == d and abs(e - b) < 1e-6 * max(1, b), (b, d, e)
print('brute force = det L_w = (1/n) prod mu_i on random connected graphs')
for n in range(1, 7):
    formula = 2 ** (2 ** n - n - 1) * math.prod(j ** math.comb(n, j) for j in range(1, n + 1))
    Q = nx.convert_node_labels_to_integers(nx.hypercube_graph(n))
    print(f'tau(Q{n}) = {tau_det(Q)}   formula {formula}')
    assert tau_det(Q) == formula
print('Petersen:', tau_det(nx.petersen_graph()), '  K_{3,4}:', tau_det(nx.complete_bipartite_graph(3, 4)), '= 3^3 * 4^2 =', 3 ** 3 * 4 ** 2)

In [ ]:
# Directed matrix-tree theorem on random multidigraphs (arcs as a list, loops and repeated arcs allowed).
def arborescences_brute(n, arcs, w):
    out = defaultdict(list)
    for a, (u, v) in enumerate(arcs): out[u].append(a)
    others = [u for u in range(n) if u != w]
    count = 0
    for choice in itertools.product(*[out[u] for u in others]):
        head = {u: arcs[a][1] for u, a in zip(others, choice)}
        ok = True
        for u in others:
            seen = set(); x = u
            while x != w:
                if x in seen: ok = False; break
                seen.add(x); x = head[x]
            if not ok: break
        count += ok
    return count
def arborescences_det(n, arcs, w):
    L = np.zeros((n, n), dtype=int)
    for u, v in arcs: L[u, u] += 1; L[u, v] -= 1
    keep = [i for i in range(n) if i != w]
    return bareiss_det(L[np.ix_(keep, keep)])
for trial in range(300):
    n = random.randint(2, 5)
    arcs = [(random.randrange(n), random.randrange(n)) for _ in range(random.randint(n, 3 * n))]
    w = random.randrange(n)
    assert arborescences_brute(n, arcs, w) == arborescences_det(n, arcs, w)
ex95 = [(0, 1), (1, 2), (2, 0), (0, 2), (2, 1), (1, 0), (0, 1)]
print('300 random directed multigraphs: brute force = det L_w.  Exercise 9.5:', [arborescences_det(3, ex95, w) for w in range(3)])

### P9.3 Uniform spanning trees

Aldous and Broder (1990) proved that the first-entrance edges of a random walk form a *uniform* spanning tree; the proof uses the Markov chain tree theorem, a cousin of Theorem 9.4 for stationary distributions. We check uniformity empirically.

In [ ]:
def aldous_broder(G, start=0):
    nbrs = {v: list(G[v]) for v in G}
    seen = {start}; x = start; T = []
    while len(seen) < len(nbrs):
        y = random.choice(nbrs[x])
        if y not in seen: seen.add(y); T.append(tuple(sorted((x, y))))
        x = y
    return frozenset(T)
for name, G, N in [('K4', nx.complete_graph(4), 32000), ('3x3 grid', nx.convert_node_labels_to_integers(nx.grid_2d_graph(3, 3)), 96000)]:
    cnt = Counter(aldous_broder(G) for _ in range(N))
    k = tau_det(G); exp = N / k
    chi2 = sum((c - exp) ** 2 / exp for c in cnt.values()) + (k - len(cnt)) * exp
    print(f'{name}: {len(cnt)} distinct trees seen out of tau = {k};  chi-square = {chi2:.1f} with {k-1} degrees of freedom')

A chi-square statistic close to its number of degrees of freedom is what uniform sampling predicts (its standard deviation is about $\sqrt{2(k-1)}$).

### P9.4 Eulerian circuits

In [ ]:
def ec_brute(n, arcs):
    # count Eulerian circuits up to rotation = Eulerian trails starting with arc 0 (all arcs distinct objects)
    out = defaultdict(list)
    for a, (u, v) in enumerate(arcs): out[u].append(a)
    m = len(arcs); used = [False] * m; used[0] = True
    def rec(x, k):                       # at vertex x, k arcs used
        if k == m: return 1 if x == arcs[0][0] else 0
        total = 0
        for a in out[x]:
            if not used[a]:
                used[a] = True; total += rec(arcs[a][1], k + 1); used[a] = False
        return total
    return rec(arcs[0][1], 1)
def random_eulerian(n, cycles):
    arcs = []
    for _ in range(cycles):
        k = random.randint(2, n); cyc = random.sample(range(n), k)
        arcs += [(cyc[i], cyc[(i + 1) % k]) for i in range(k)]
    return arcs
def best(n, arcs):
    outdeg = Counter(u for u, v in arcs)
    return arborescences_det(n, arcs, 0) * math.prod(math.factorial(outdeg[v] - 1) for v in range(n))
checked = 0
for trial in range(400):
    n = random.randint(2, 5); arcs = random_eulerian(n, random.randint(2, 4))
    D = nx.MultiDiGraph(arcs)
    if D.number_of_nodes() < n or not nx.is_strongly_connected(D): continue
    assert ec_brute(n, arcs) == best(n, arcs); checked += 1
print(checked, 'random Eulerian multidigraphs: backtracking count = BEST formula')
def debruijn_count_brute(q, n):
    # all words of length q^n whose cyclic windows of length n are distinct, counted up to rotation
    L = q ** n; found = set()
    def rec(seq, seen):
        if len(seq) == L:
            tail = seq[L - n + 1:] + seq[:n - 1]                  # windows that wrap around
            wrap = {tuple(tail[i:i + n]) for i in range(n - 1)}
            if len(wrap) == n - 1 and not (wrap & seen): found.add(tuple(seq))
            return
        for c in range(q):
            nxt = seq + [c]
            if len(nxt) >= n:
                w = tuple(nxt[-n:])
                if w in seen: continue
                rec(nxt, seen | {w})
            else:
                rec(nxt, seen)
    rec([], frozenset())
    return len({min(s[i:] + s[:i] for i in range(L)) for s in found})
for q, n in [(2, 2), (2, 3), (2, 4), (3, 2), (2, 5)]:
    formula = Fraction(math.factorial(q) ** (q ** (n - 1)), q ** n)
    print(f'q={q}, n={n}: {debruijn_count_brute(q, n)} de Bruijn sequences, formula {formula}')

### P9.5 Cheeger's inequality

In [ ]:
def exact_h(G):
    n = G.number_of_nodes(); E = np.array(list(G.edges)); best = math.inf
    for mask in range(1, 1 << n):
        s = bin(mask).count('1')
        if s > n // 2: continue
        inS = np.array([(mask >> v) & 1 for v in range(n)], dtype=bool)
        cut = np.count_nonzero(inS[E[:, 0]] != inS[E[:, 1]])
        best = min(best, cut / s)
    return best
def sweep(G, vec):
    n = G.number_of_nodes(); order = np.argsort(vec); best = math.inf; inS = np.zeros(n, bool); E = np.array(list(G.edges))
    for k in range(1, n):
        inS[order[k - 1]] = True
        cut = np.count_nonzero(inS[E[:, 0]] != inS[E[:, 1]])
        best = min(best, cut / min(k, n - k))
    return best
rows = []
t0 = time.time()
for trial in range(12):
    n = random.choice([10, 12, 14, 16]); G = nx.random_regular_graph(3, n, seed=trial)
    if not nx.is_connected(G): continue
    w, V = np.linalg.eigh(lap(G)); mu2 = w[1]
    h = exact_h(G); s = sweep(G, V[:, 1])
    assert mu2 / 2 <= h + 1e-9 <= math.sqrt(6 * mu2) + 1e-9 and h <= s + 1e-9 <= math.sqrt(6 * mu2) + 1e-9
    rows.append((n, mu2 / 2, h, s, math.sqrt(6 * mu2)))
print(' n   mu2/2      h(G)    sweep   sqrt(2d mu2)')
for r in rows: print(f'{r[0]:2d}  {r[1]:.3f}   {r[2]:.3f}   {r[3]:.3f}   {r[4]:.3f}')
print(f'({time.time()-t0:.1f} s)')

The exact $h$ always sits between the two bounds, and the sweep set is often optimal or close: the right inequality of the theorem is about the sweep, not only about $h$.

### P9.6 Spectral clustering

In [ ]:
def sbm(n, p, q):
    half = n // 2; G = nx.Graph(); G.add_nodes_from(range(n))
    A = rng.random((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            same = (i < half) == (j < half)
            if A[i, j] < (p if same else q): G.add_edge(i, j)
    return G
n, p = 400, 0.15
qs = [0.01, 0.03, 0.05, 0.07, 0.09, 0.11, 0.13, 0.15]
acc = []
for q in qs:
    a = []
    for rep in range(5):
        G = sbm(n, p, q)
        nodes = sorted(G.nodes); w, V = np.linalg.eigh(nx.laplacian_matrix(G, nodelist=nodes).toarray().astype(float))
        guess = V[:, 1] > np.median(V[:, 1]); truth = np.array([v < n // 2 for v in nodes])
        a.append(max(np.mean(guess == truth), np.mean(guess != truth)))
    acc.append(np.mean(a)); print(f'q = {q:.3f}: accuracy {acc[-1]:.3f}')
plt.figure(figsize=(4.5, 3)); plt.plot(qs, acc, 'o-'); plt.axhline(0.5, color='gray', ls=':')
plt.xlabel('q (across)'); plt.ylabel('fraction recovered'); plt.title(f'Fiedler vector on SBM, n={n}, p={p}'); plt.show()

Recovery is nearly perfect while $q$ is small compared with $p$ and degrades to a coin toss as $q\to p$. (In sparse graphs the plain Laplacian does worse: its low eigenvectors tend to localise on small dangling trees, which is why practitioners normalise it.) For the sparse version of this model there is a sharp threshold, $(a-b)^2>2(a+b)$ when $p=\frac an$, $q=\frac bn$, found by physicists (Decelle, Krzakala, Moore and Zdeborová, 2011) and proved by Mossel, Neeman, Sly and Massoulié: below it no algorithm can do better than chance.

### P9.7 How good are random expanders?

In [ ]:
print(' d     n    lambda   2 sqrt(d-1)')
for d in [3, 4, 6, 10]:
    for n in [200, 1000]:
        G = nx.random_regular_graph(d, n, seed=d * n)
        ev = np.linalg.eigvalsh(nx.to_numpy_array(G))
        lam = max(abs(ev[0]), abs(ev[-2]))
        print(f'{d:2d}  {n:5d}   {lam:.3f}   {2*math.sqrt(d-1):.3f}')
# Expander mixing lemma on random sets.
d, n = 6, 1000
G = nx.random_regular_graph(d, n, seed=1); A = nx.to_numpy_array(G)
ev = np.linalg.eigvalsh(A); lam = max(abs(ev[0]), abs(ev[-2])); worst = 0
for trial in range(300):
    s, t = random.randint(1, n - 1), random.randint(1, n - 1)
    S = rng.choice(n, s, replace=False); T = rng.choice(n, t, replace=False)
    e = A[np.ix_(S, T)].sum(); dev = abs(e - d * s * t / n)
    bound = lam * math.sqrt(s * t * (1 - s / n) * (1 - t / n))
    assert dev <= bound + 1e-9; worst = max(worst, dev / bound)
print(f'mixing lemma holds on 300 random pairs (S,T); largest deviation/bound = {worst:.3f}')
# random sets are far from the worst case; the worst sets are built from eigenvectors: S = {top coordinates of psi_2}
w, V = np.linalg.eigh(A); S = np.argsort(V[:, -2])[-n // 4:]
e = A[np.ix_(S, S)].sum(); s = len(S)
print(f'S = top quarter of the second eigenvector: e(S,S) = {e:.0f}, expected d s^2/n = {d*s*s/n:.0f}, bound on the deviation {lam*s*(1-s/n):.0f}')

In [ ]:
# Alon-Boppana with actual distances: find two edges as far apart as possible, take the largest k allowed.
def edge_distance(G, e, f, dist):
    return min(dist[a][b] for a in e for b in f)
for d, n in [(3, 400), (4, 600)]:
    G = nx.random_regular_graph(d, n, seed=7)
    dist = dict(nx.all_pairs_shortest_path_length(G))
    E = list(G.edges); e0 = E[0]
    far = max(E, key=lambda f: edge_distance(G, e0, f, dist))
    D = edge_distance(G, e0, far, dist); k = (D - 2) // 2
    bound = 2 * math.sqrt(d - 1) - (2 * math.sqrt(d - 1) - 1) / (k + 1)
    lam2 = np.linalg.eigvalsh(nx.to_numpy_array(G))[-2]
    print(f'd={d}, n={n}: two edges at distance {D}, k={k}: lambda_2 = {lam2:.3f} >= {bound:.3f} (2 sqrt(d-1) = {2*math.sqrt(d-1):.3f})')
    assert lam2 >= bound

Random regular graphs have $\lambda$ within a few percent of $2\sqrt{d-1}$, as Friedman's theorem predicts, while the Alon–Boppana bound with realistic distances is weaker but already within striking range: the truth for random graphs is squeezed between the two.

### P9.8 Random walks forget

In [ ]:
n = 200
for name, G in [('random 4-regular', nx.random_regular_graph(4, n, seed=3)), ('cycle C_201', nx.cycle_graph(n + 1))]:
    A = nx.to_numpy_array(G); N = A.shape[0]; d = int(A.sum(1)[0])
    ev = np.linalg.eigvalsh(A); lam = max(abs(ev[0]), abs(ev[-2]))
    p = np.zeros(N); p[0] = 1; u = np.ones(N) / N; ts, dist = [], []
    for t in range(0, 61):
        ts.append(t); dist.append(np.linalg.norm(p - u)); p = A @ p / d
    assert all(dist[t] <= (lam / d) ** t + 1e-12 for t in ts)
    plt.semilogy(ts, dist, label=f'{name} (lambda/d = {lam/d:.3f})')
plt.xlabel('t'); plt.ylabel('||p_t - u||'); plt.legend(); plt.title('distance to uniform'); plt.show()

The expander forgets its starting point after a few dozen steps; the cycle (odd length, so not bipartite) has $\frac\lambda d=\cos\frac{\pi}{201}$, and needs about $n^2$ steps. This gap is why expanders are used to save random bits (Tome C3): a walk of $t$ steps on an expander uses $\log n+t\log d$ random bits but behaves almost like $t$ independent samples.